In [4]:
import pandas as pd
import numpy as np

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# Neural Network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("✅ Libraries loaded")

✅ Libraries loaded


In [5]:
import pandas as pd
from google.colab import files
upload=files.upload()


Saving Data_Test.csv to Data_Test.csv
Saving Data_Train.csv to Data_Train.csv


In [6]:
train_df = pd.read_csv('Data_Train.csv', encoding='latin1')
test_df = pd.read_csv('Data_Test.csv', encoding='latin1')

print(train_df.head())
print(train_df.columns)

                                               STORY  SECTION
0  But the most painful was the huge reversal in ...        3
1  How formidable is the opposition alliance amon...        0
2  Most Asian currencies were trading lower today...        3
3  If you want to answer any question, click on ...        1
4  In global markets, gold prices edged up today ...        3
Index(['STORY', 'SECTION'], dtype='object')


In [7]:
# Remove missing values
train_df = train_df.dropna()

# Rename for clarity (optional)
train_df = train_df.rename(columns={"STORY": "text", "SECTION": "label"})

X = train_df['text']
y = train_df['label']

In [8]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)

Classes: [0 1 2 3]


In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

In [10]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print("Text vectorized")

Text vectorized


In [11]:
svm = SVC()
svm.fit(X_train_tfidf, y_train)

svm_preds = svm.predict(X_val_tfidf)

print("SVM Accuracy:", accuracy_score(y_val, svm_preds))
print(classification_report(y_val, svm_preds))

SVM Accuracy: 0.9705111402359109
              precision    recall  f1-score   support

           0       0.98      0.94      0.96       323
           1       0.97      0.98      0.98       549
           2       0.97      0.98      0.97       402
           3       0.97      0.97      0.97       252

    accuracy                           0.97      1526
   macro avg       0.97      0.97      0.97      1526
weighted avg       0.97      0.97      0.97      1526



In [12]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train_tfidf, y_train)

rf_preds = rf.predict(X_val_tfidf)

print("RF Accuracy:", accuracy_score(y_val, rf_preds))
print(classification_report(y_val, rf_preds))

RF Accuracy: 0.9338138925294889
              precision    recall  f1-score   support

           0       0.96      0.90      0.93       323
           1       0.95      0.95      0.95       549
           2       0.89      0.98      0.93       402
           3       0.96      0.87      0.91       252

    accuracy                           0.93      1526
   macro avg       0.94      0.92      0.93      1526
weighted avg       0.94      0.93      0.93      1526



In [13]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train_tfidf, y_train)

xgb_preds = xgb.predict(X_val_tfidf)

print("XGB Accuracy:", accuracy_score(y_val, xgb_preds))
print(classification_report(y_val, xgb_preds))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:44:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGB Accuracy: 0.9442988204456094
              precision    recall  f1-score   support

           0       0.96      0.90      0.93       323
           1       0.95      0.95      0.95       549
           2       0.92      0.97      0.94       402
           3       0.95      0.94      0.94       252

    accuracy                           0.94      1526
   macro avg       0.95      0.94      0.94      1526
weighted avg       0.94      0.94      0.94      1526



In [14]:
# Convert sparse → dense
X_train_dense = X_train_tfidf.toarray()
X_val_dense = X_val_tfidf.toarray()

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_dense.shape[1],)),
    Dense(64, activation='relu'),
    Dense(len(np.unique(y_encoded)), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(X_train_dense, y_train, epochs=5, batch_size=32)

nn_preds = np.argmax(model.predict(X_val_dense), axis=1)

print("NN Accuracy:", accuracy_score(y_val, nn_preds))
print(classification_report(y_val, nn_preds))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8707 - loss: 0.4341
Epoch 2/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9902 - loss: 0.0378
Epoch 3/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9966 - loss: 0.0115
Epoch 4/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9974 - loss: 0.0081
Epoch 5/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9974 - loss: 0.0075
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
NN Accuracy: 0.9724770642201835
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       323
           1       0.98      0.98      0.98       549
           2       0.97      0.98      0.97       402
           3       0.96      0.98      0.97       252

    accuracy                           0.97      1526
   macro avg       0.97      0.97      0.97      1526
weighted avg       0.97      0.97      0.97      1526



In [15]:
!pip install transformers torch -q

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

In [16]:
bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_name,
    num_labels=len(le.classes_)
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def tokenize(texts, tokenizer):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )
bert_train_enc = tokenize(X_train, bert_tokenizer)
bert_val_enc = tokenize(X_val, bert_tokenizer)

In [18]:

class NLPDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.values  # <-- important

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [19]:
bert_train_dataset = NLPDataset(bert_train_enc, pd.Series(y_train))
bert_val_dataset = NLPDataset(bert_val_enc, pd.Series(y_val))

In [20]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,   # keep 1 (faster)
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=100,
    save_strategy="no"
)

In [21]:
from transformers import Trainer

bert_trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_dataset,
    eval_dataset=bert_val_dataset
)

bert_trainer.train()

Step,Training Loss
100,0.393420
200,0.143841
300,0.173323
400,0.144158
500,0.108915
600,0.150457
700,0.052013


TrainOutput(global_step=763, training_loss=0.1621565181337209, metrics={'train_runtime': 147.7123, 'train_samples_per_second': 41.31, 'train_steps_per_second': 5.165, 'total_flos': 401383122536448.0, 'train_loss': 0.1621565181337209, 'epoch': 1.0})

In [22]:
bert_outputs = bert_trainer.predict(bert_val_dataset)

bert_preds = np.argmax(bert_outputs.predictions, axis=1)

print("BERT Accuracy:", accuracy_score(y_val, bert_preds))

BERT Accuracy: 0.9823066841415465


In [23]:
roberta_name = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_name,
    num_labels=len(le.classes_)
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
roberta_train_enc = tokenize(X_train, roberta_tokenizer)
roberta_val_enc = tokenize(X_val, roberta_tokenizer)

In [26]:
roberta_train_dataset = NLPDataset(roberta_train_enc, pd.Series(y_train))
roberta_val_dataset = NLPDataset(roberta_val_enc, pd.Series(y_val))

In [27]:
roberta_trainer = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=roberta_train_dataset,
    eval_dataset=roberta_val_dataset
)

roberta_trainer.train()

Step,Training Loss
100,0.466981
200,0.168328
300,0.193643
400,0.193830
500,0.148993
600,0.136645
700,0.046495


TrainOutput(global_step=763, training_loss=0.18921437269738273, metrics={'train_runtime': 155.3162, 'train_samples_per_second': 39.288, 'train_steps_per_second': 4.913, 'total_flos': 401383122536448.0, 'train_loss': 0.18921437269738273, 'epoch': 1.0})

In [28]:
roberta_outputs = roberta_trainer.predict(roberta_val_dataset)

roberta_preds = np.argmax(roberta_outputs.predictions, axis=1)

print("RoBERTa Accuracy:", accuracy_score(y_val, roberta_preds))

RoBERTa Accuracy: 0.971821756225426


In [29]:
ensemble_preds = np.vstack([
    svm_preds,
    rf_preds,
    xgb_preds,
    nn_preds,
    bert_preds,
    roberta_preds
])

# Majority voting
from scipy.stats import mode
ensemble_preds = mode(ensemble_preds, axis=0).mode.flatten()

print("Ensemble Accuracy:", accuracy_score(y_val, ensemble_preds))

Ensemble Accuracy: 0.9737876802096985


In [30]:
results = pd.DataFrame({
    "Model": [
        "SVM",
        "Random Forest",
        "XGBoost",
        "Neural Network",
        "BERT",
        "RoBERTa",
        "Ensemble"
    ],
    "Accuracy": [
        accuracy_score(y_val, svm_preds),
        accuracy_score(y_val, rf_preds),
        accuracy_score(y_val, xgb_preds),
        accuracy_score(y_val, nn_preds),
        accuracy_score(y_val, bert_preds),
        accuracy_score(y_val, roberta_preds),
        accuracy_score(y_val, ensemble_preds)
    ]
})

# Sort (optional but nice)
results = results.sort_values(by="Accuracy", ascending=False)

print("\n📊 MODEL COMPARISON TABLE")
print(results)

# Save for report
results.to_csv("model_results.csv", index=False)


📊 MODEL COMPARISON TABLE
            Model  Accuracy
4            BERT  0.982307
6        Ensemble  0.973788
3  Neural Network  0.972477
5         RoBERTa  0.971822
0             SVM  0.970511
2         XGBoost  0.944299
1   Random Forest  0.933814


In [33]:
# Preprocess test data
test_df = test_df.rename(columns={"STORY": "text"})
X_test = test_df['text']
X_test_tfidf = vectorizer.transform(X_test)

# Classical models
svm_test = svm.predict(X_test_tfidf)
rf_test = rf.predict(X_test_tfidf)
xgb_test = xgb.predict(X_test_tfidf)

# NN
X_test_dense = X_test_tfidf.toarray()
nn_test = np.argmax(model.predict(X_test_dense), axis=1)

# BERT
bert_test_enc = tokenize(X_test, bert_tokenizer)
bert_test_dataset = NLPDataset(bert_test_enc, pd.Series([0]*len(X_test)))

bert_test_preds = np.argmax(
    bert_trainer.predict(bert_test_dataset).predictions, axis=1
)

# RoBERTa
roberta_test_enc = tokenize(X_test, roberta_tokenizer)
roberta_test_dataset = NLPDataset(roberta_test_enc, pd.Series([0]*len(X_test)))

roberta_test_preds = np.argmax(
    roberta_trainer.predict(roberta_test_dataset).predictions, axis=1
)

# Combine
from scipy.stats import mode

all_preds = np.vstack([
    svm_test,
    rf_test,
    xgb_test,
    nn_test,
    bert_test_preds,
    roberta_test_preds
])

final_preds = mode(all_preds, axis=0).mode.flatten()
final_labels = le.inverse_transform(final_preds)
test_df['Predicted_Section'] = final_labels

test_df.to_csv("final_predictions.csv", index=False)

print("Final predictions saved as final_predictions.csv")

86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


Final predictions saved as final_predictions.csv
